In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pathlib

from pmw_analysis.constants import ArgSurfaceType, COLUMN_TEMP_2M_INDEX
from pmw_analysis.constants import DIR_PMW_ANALYSIS, TC_COLUMNS, ArgTransform
from pmw_analysis.quantization.script import get_transformation_function

arg_transform = ArgTransform.V4
arg_surface_type = ArgSurfaceType.LAND
df_dir_path = pathlib.Path(DIR_PMW_ANALYSIS) / arg_transform.value / arg_surface_type.value

transform = get_transformation_function(arg_transform)
quant_columns = transform(TC_COLUMNS)

In [3]:
from pmw_analysis.constants import COLUMN_TIME, COLUMN_LON, COLUMN_LAT, COLUMN_GPM_ID, COLUMN_GPM_CROSS_TRACK_ID
import polars as pl

df_peaks_path = df_dir_path / "final_rarest_nbr_count_0_m_3_peaks.parquet"
df_peaks = pl.read_parquet(df_peaks_path)

ids = set(df_peaks.select(pl.concat_str([pl.col(COLUMN_GPM_ID), pl.col(COLUMN_GPM_CROSS_TRACK_ID)], separator=" ")).to_series())

peak_points = df_peaks.filter(pl.col("peaks_lon_lat").is_not_null()).group_by("peaks_lon_lat").agg(
    [pl.col(COLUMN_LON).mean(), pl.col(COLUMN_LAT).mean()])

In [4]:
from pmw_analysis.constants import DIR_IMAGES
from pmw_analysis.utils.io import combine_paths, file_to_dir

images_dir = combine_paths(path_base=pathlib.Path("..") / DIR_IMAGES, path_rel=file_to_dir(df_peaks_path), path_rel_base=DIR_PMW_ANALYSIS) / "ts"
images_dir.mkdir(parents=True, exist_ok=True)

In [5]:
import datetime
import gpm.bucket
from pmw_analysis.processing.filter import filter_by_value_range
from pmw_analysis.constants import DIR_BUCKET
from examples.signatures_evolution import _detect_peaks, _take_k_peaks

DISTANCE = 10000

bin_sizes_time = {COLUMN_TIME: datetime.timedelta(days=4)}
bin_ranges_time = {}

ts_list = []
ts_newest_list = []
ts_newest_peaks_list = []
ts_dpr_list = []

for peak_label, lon, lat in peak_points.iter_rows():
    point = (lon, lat)
    ts = gpm.bucket.read(bucket_dir=DIR_BUCKET, point=point, distance=DISTANCE,
                         columns=[COLUMN_TIME, COLUMN_LON, COLUMN_LAT] + TC_COLUMNS + [COLUMN_GPM_ID, COLUMN_GPM_CROSS_TRACK_ID] + [COLUMN_TEMP_2M_INDEX])
    ts_dpr = gpm.bucket.read(bucket_dir="/ltenas8/data/GPM_Buckets/DPR_RainySurface", point=point, distance=DISTANCE,
                             backend="polars", columns=[COLUMN_LON, COLUMN_LAT, COLUMN_TIME, "REFC_Ka", "snowIceCover"])

    start_time = ts[COLUMN_TIME].min()  # ts[COLUMN_TIME].max() - datetime.timedelta(days=730)
    end_time = ts[COLUMN_TIME].max()
    ts = filter_by_value_range(ts, COLUMN_TIME, (start_time, end_time))
    ts_dpr = filter_by_value_range(ts_dpr, COLUMN_TIME, (start_time, end_time))

    ts = transform(ts)
    ts = ts.with_columns(pl.concat_str([pl.col(COLUMN_GPM_ID), pl.col(COLUMN_GPM_CROSS_TRACK_ID)], separator=" ").alias("full_id"))
    ts_newest = ts.filter(pl.col("full_id").is_in(ids))

    peaks_time = _detect_peaks(ts_newest[[COLUMN_TIME]], bin_sizes_time, bin_ranges_time, images_dir, verbose=False)
    peaks_time_k = _take_k_peaks(peaks_time, 1)
    ts_newest_peaks = ts_newest.filter(peaks_time_k["peaks"].is_not_null())

    ts_list.append(ts)
    ts_newest_list.append(ts_newest)
    ts_newest_peaks_list.append(ts_newest_peaks)
    ts_dpr_list.append(ts_dpr)

In [46]:
peaks = None
for ts_newest_peaks in ts_newest_peaks_list:
    peak = ts_newest_peaks[(ts_newest_peaks["gpm_cross_track_id"] - 110).abs().arg_min()]
    peaks = peak if peaks is None else peaks.extend(peak)
peaks.write_parquet(df_dir_path / f"{df_peaks_path.stem}_to_analyze.parquet")

In [8]:
import matplotlib.pyplot as plt
import numpy as np
import datetime
from examples.signatures_evolution import _detect_peaks, _take_k_peaks

bin_sizes_time = {COLUMN_TIME: datetime.timedelta(days=4)}
bin_ranges_time = {}

n = len(quant_columns)

for idx_col, (ts, ts_newest, ts_newest_peaks, ts_dpr) in enumerate(zip(ts_list, ts_newest_list, ts_newest_peaks_list, ts_dpr_list)):
    _, axes = plt.subplots(n, 1, figsize=(20, 4 * n), dpi=300)

    if n == 1:
        axes = np.array([axes])

    name = f"lon={peak_points[idx_col]['lon'].item():.2f}_lat={peak_points[idx_col]['lat'].item():.2f}_ts"
    for idx_row, col in enumerate(quant_columns):
        # TODO: think about removing this assertion
        # assert ts[feature_col].isna().sum() == 0

        ax = axes[idx_row]
        plt.sca(ax)
        # plt.scatter(ts_dpr[COLUMN_TIME], ts_dpr["REFC_Ka"], marker="x", c=ts_dpr["snowIceCover"], s=1)
        # plt.ylabel("%")
        #
        # ax_twinx = ax.twinx()
        plt.plot(ts[COLUMN_TIME], ts[COLUMN_TEMP_2M_INDEX], linestyle="--", linewidth=1, alpha=0.2)
        plt.ylabel("Degrees")

        ax_twinx = ax.twinx()
        plt.scatter(ts[COLUMN_TIME], ts[col], c="blue", s=2)
        plt.scatter(ts_newest[COLUMN_TIME], ts_newest[col], c="orange", s=2)
        plt.scatter(ts_newest_peaks[COLUMN_TIME], ts_newest_peaks[col], c="red", s=2, marker='x')

        plt.xticks(rotation=90)
        plt.ylabel("[K]")

        plt.title(f"{col} ({name})")

    plt.tight_layout()
    plt.savefig(images_dir / f"{name}.png")
    # plt.show()
    plt.clf()

<Figure size 6000x6000 with 0 Axes>

<Figure size 6000x6000 with 0 Axes>

<Figure size 6000x6000 with 0 Axes>

<Figure size 6000x6000 with 0 Axes>

<Figure size 6000x6000 with 0 Axes>

...

<Figure size 6000x6000 with 0 Axes>

<Figure size 6000x6000 with 0 Axes>

<Figure size 6000x6000 with 0 Axes>

<Figure size 6000x6000 with 0 Axes>

<Figure size 6000x6000 with 0 Axes>